## 0. Setup
Mục tiêu: import các thư viện cần thiết

In [1]:
!pip install sqlalchemy psycopg2-binary pandas
import pandas as pd
from sqlalchemy import create_engine, inspect, text
!pip install flask openai python-dotenv
import numpy as np

RANDOM_SEED = 45
np.random.seed(RANDOM_SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 20.4 MB/s eta 0:00:00


##1. Problem & Metrics

**Bài toán:** Chuyển ngôn ngữ tự nhiên (text) qua câu lệnh truy vấn (SQL) (Text to SQL)

**Metrics sử dụng:**

*   Valid SQL Rate (%): Tỷ lệ câu lệnh SQL chạy được trên database mà không báo lỗi cú pháp.

*   Execution Accuracy (EX) (%): Tỷ lệ kết quả truy vấn của Chatbot trùng khớp với kết quả chuẩn (Ground Truth)


## 2. Load data
*  Import data (Olist) vào PostgreSQL
*  Xây dựng bộ câu hỏi và import data bộ câu hỏi vào PostgreSQL
*  Kết nối data đến Colab
*  Khai phá dữ liệu

In [2]:
DATABASE_URL = "YOUR_SUPABASE_URL"
engine = create_engine(DATABASE_URL)

inspector = inspect(engine)

tables = inspector.get_table_names()

with engine.connect() as conn:
    for table in tables:
        # đếm số dòng
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        row_count = result.scalar()

        # đếm số cột
        col_count = len(inspector.get_columns(table))

        print(f"{table} -> shape: ({row_count}, {col_count})")

metadata_enhanced_schema -> shape: (52, 13)
metadata_kpi_definitions -> shape: (8, 7)
metadata_question -> shape: (90, 3)
metadata_question_few_shot -> shape: (8, 3)
metadata_business_rules -> shape: (21, 8)
products -> shape: (32951, 9)
order_items -> shape: (112650, 7)
orders -> shape: (99441, 8)
order_reviews -> shape: (99224, 7)
order_payments -> shape: (103886, 5)
product_category_name_translation -> shape: (71, 2)
sellers -> shape: (3095, 4)
customers -> shape: (99441, 5)


In [3]:
# đọc bảng questions vào dataframe
questions_df = pd.read_sql("SELECT * FROM metadata_question ORDER BY question_vie", engine)
print("shape:", questions_df.shape)
questions_df.head(30)

shape: (90, 3)


,question_vie,question_level,ground_truth_sql
0,Bang nào có ít người bán nhất?,Trung bình,"SELECT seller_state, COUNT(DISTINCT seller_id)..."
1,Bang nào có nhiều người bán nhất?,Trung bình,"SELECT seller_state, COUNT(DISTINCT seller_id)..."
2,Đếm số lượng bang chỉ có đúng 1 người bán?,Trung bình,SELECT COUNT(*) AS total_states FROM (SELECT s...
3,Đếm số lượng bang có ít nhất 10 người bán?,Trung bình,SELECT COUNT(*) AS total_states FROM ( SELECT ...
4,Đếm số lượng bang có số lượng người bán bằng đ...,Trung bình,SELECT COUNT(*) AS total_states FROM (SELECT s...
5,Đếm số lượng người bán (sellers) trên hệ thống?,Dễ,SELECT COUNT(DISTINCT seller_id) AS total_sell...
6,Đếm số lượng người bán không thuộc bang 'SP'?,Dễ,SELECT COUNT(DISTINCT seller_id) AS total_sell...
7,Đếm số lượng người bán ở bang 'SP'?,Dễ,SELECT COUNT(DISTINCT seller_id) AS total_sell...
8,Đếm số lượng người bán ở thành phố có tên thàn...,Dễ,SELECT COUNT(DISTINCT seller_id) AS total_sell...
9,Đếm số lượng người bán ở thành phố sao paulo?,Dễ,SELECT COUNT(DISTINCT seller_id) AS total_sell...


## 3. Train/Validation/Test Split cho dữ liệu bộ câu hỏi

In [ ]:
#lấy các câu tuyển chọn từ bảng question_few_shot để đặt vào prompt train_df

train_df = pd.read_sql("SELECT * FROM metadata_question_few_shot ORDER BY question_vie", engine)

# chia bảng câu hỏi chính thành 20% cho val & 80% cho test

from sklearn.model_selection import train_test_split

val_df, test_df = train_test_split(
    questions_df,
    test_size=0.8,
    random_state=RANDOM_SEED,
    stratify=questions_df["question_level"]
)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (8, 3)
Val shape: (18, 3)
Test shape: (72, 3)


In [ ]:
#chuyển train df thành text cho promp
def build_fewshot_examples(train_df, max_examples=10):
    examples_text = ""

    # lấy top N examples (có thể random nếu muốn)
    sampled_df = train_df.head(10)

    for _, row in sampled_df.iterrows():
        question = row["question_vie"]
        sql = row["ground_truth_sql"]

        examples_text += f"Q: {question}\n"
        examples_text += f"SQL:\n{sql}\n\n"

    return examples_text

example_text = build_fewshot_examples(train_df, max_examples=5)

print(example_text)

Q: Đếm số lượng thành phố có ít nhất 5 người bán?
SQL:
SELECT COUNT(*) AS total_cities FROM (SELECT seller_city FROM sellers GROUP BY seller_city HAVING COUNT(DISTINCT seller_id) >= 5) t;

Q: Doanh thu của người bán có mã '3442f8959a84dea7ee197c632cb2df15' là bao nhiêu?
SQL:
SELECT ROUND(SUM(oi.price), 2) AS total_revenue FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND oi.seller_id = '3442f8959a84dea7ee197c632cb2df15';

Q: doanh thu theo từng danh mục sản phẩm trong năm 2017 là bao nhiêu?
SQL:
SELECT t.product_category_name_english AS category, ROUND(SUM(oi.price +oi.freight_value),2) AS revenue FROM order_items oi JOIN orders o ON oi.order_id=o.order_id JOIN products pr ON oi.product_id=pr.product_id JOIN product_category_name_translation t ON pr.product_category_name=t.product_category_name WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp)=2017 GROUP BY t.product_category_name_english

Q: giá trị đơn hàng trung bình của những đơn

## 4. Representation (database schema)
Biểu diễn cấu trúc database để model hiểu

#### Raw schema

In [ ]:
# lấy schema thô
inspector = inspect(engine)

tables = [t for t in inspector.get_table_names() if not t.startswith("metadata_")]

raw_schema = {}

for table in tables:
    columns = inspector.get_columns(table)
    column_info = [
        f"{col['name']} ({str(col['type'])})"
        for col in columns
    ]
    raw_schema[table] = column_info

# convert sang text
def format_schema(raw_schema):
    schema_text = ""

    for table, columns in raw_schema.items():
        schema_text += f"Table: {table}\n"
        for col in columns:
            schema_text += f"- {col}\n"
        schema_text += "\n"

    return schema_text

raw_schema_text = format_schema(raw_schema)

print(raw_schema_text)

Table: products
- product_id (VARCHAR)
- product_category_name (VARCHAR)
- product_name_lenght (BIGINT)
- product_description_lenght (BIGINT)
- product_photos_qty (BIGINT)
- product_weight_g (BIGINT)
- product_length_cm (BIGINT)
- product_height_cm (BIGINT)
- product_width_cm (BIGINT)

Table: order_items
- order_id (VARCHAR)
- order_item_id (INTEGER)
- product_id (VARCHAR)
- seller_id (VARCHAR)
- shipping_limit_date (TIMESTAMP)
- price (NUMERIC)
- freight_value (NUMERIC)

Table: orders
- order_id (VARCHAR)
- customer_id (VARCHAR)
- order_status (VARCHAR)
- order_purchase_timestamp (TIMESTAMP)
- order_approved_at (TIMESTAMP)
- order_delivered_carrier_date (TIMESTAMP)
- order_delivered_customer_date (TIMESTAMP)
- order_estimated_delivery_date (TIMESTAMP)

Table: order_reviews
- review_id (VARCHAR)
- order_id (VARCHAR)
- review_score (INTEGER)
- review_comment_title (VARCHAR)
- review_comment_message (TEXT)
- review_creation_date (TIMESTAMP)
- review_answer_timestamp (TIMESTAMP)

Table: o

#### Enhanced Schema

In [ ]:
import pandas as pd
import re
import json

# =========================
# 1. Load schema từ DB
# =========================
query = "SELECT * FROM metadata_enhanced_schema"
df_schema = pd.read_sql(query, engine)

# =========================
# 2. Normalize dữ liệu DB
# =========================
df_schema["table_name"] = df_schema["table_name"].str.strip().str.lower()
df_schema["column_name"] = df_schema["column_name"].str.strip().str.lower()

# =========================
# 3. Clean function cho column (remove datatype)
# =========================
def clean_column(col):
    return re.sub(r"\s*\(.*?\)", "", col).strip().lower()

# =========================
# 4. Normalize raw_schema
# =========================
raw_schema_clean = {}

for table, cols in raw_schema.items():
    table_clean = table.strip().lower()
    raw_schema_clean[table_clean] = cols  # giữ nguyên col gốc để làm key

# normalize tables list
tables = [t for t in inspector.get_table_names() if not t.startswith("metadata_")]

# =========================
# 5. Build enhanced schema
# =========================
enhanced_schema = {}

for table in tables:
    if table not in raw_schema_clean:
        print(f"⚠️ Table {table} not in raw_schema")
        continue

    table_columns = df_schema[df_schema["table_name"] == table]

    #print(f"\n📊 Processing table: {table} | columns found: {len(table_columns)}")

    enhanced_schema[table] = {}

    for col in raw_schema_clean[table]:
        clean_col = clean_column(col)

        col_info = table_columns[
            table_columns["column_name"] == clean_col
        ]

        if not col_info.empty:
            row = col_info.iloc[0]

            enhanced_schema[table][col] = {
                "table_description": row.get("table_description_vi"),
                "column_name_clean": clean_col,  # 👈 thêm để debug / dùng sau
                "data_type": row.get("data_type"),
                "column_description_vi": row.get("column_description_vi"),
                "column_description_en": row.get("column_description_en"),
                "PK_key_type": row.get("PK_key_type"),
                "FK_key_type": row.get("FK_key_type"),
                "references_to": row.get("references_to"),
                "allowed_values": row.get("allowed_values"),
                "constraints": row.get("constraints"),
                "example": row.get("example"),
            }
        else:
            # print(f"❌ Missing: {table}.{clean_col}")

            enhanced_schema[table][col] = {
                "column_name_clean": clean_col,
                "data_type": None,
                "description_vi": None
            }

# =========================
# 6. Preview (tránh print quá to)
# =========================
#preview = dict(list(enhanced_schema.items())[:2])

#print("\n✅ Preview enhanced schema:")
#print(json.dumps(preview, indent=2, ensure_ascii=False))

# =========================
# Convert enhanced_schema → text
def enhanced_schema_to_text(enhanced_schema):
    schema_text = ""

    for table, cols in enhanced_schema.items():
        schema_text += f"\n### Table: {table}\n"

        # lấy table description (nếu có)
        table_desc = None
        for c in cols.values():
            if c.get("table_description"):
                table_desc = c.get("table_description")
                break

        if table_desc:
            schema_text += f"Table Description: {table_desc}\n"

        schema_text += "Columns:\n"

        for col_raw, info in cols.items():
            col = info.get("column_name_clean", col_raw)

            line = f"- {col}"

            # data type
            if info.get("data_type"):
                line += f" ({info['data_type']})"

            # description vi
            desc = info.get("column_description_vi")
            if desc:
                line += f": {desc}"

            # description en
            desc = info.get("column_description_en")
            if desc:
                line += f" ({desc})"

            # PK key
            if info.get("PK_key_type"):
                line += f" {info['PK_key_type']}"

            # FK key
            if info.get("FK_key_type"):
                line += f" | {info['FK_key_type']}"

            # reference
            if info.get("references_to"):
                line += f" | ref: {info['references_to']}"

            # allowed values
            if info.get("allowed_values"):
                values = str(info["allowed_values"])
                line += f" | allowed values: {values}"

            # constraints
            if info.get("constraints"):
                values = str(info["constraints"])
                line += f" | constraints: {values}"

            # example
            if info.get("example"):
                values = str(info["example"])
                line += f" | example: {values}"

            schema_text += line + "\n"

    return schema_text.strip()

#Generate schema_text

enhanced_schema_text = enhanced_schema_to_text(enhanced_schema)

#print("\n✅ Schema text for prompt:\n")
print(enhanced_schema_text)

### Table: products
Table Description: chứa thông tin về đặc điểm sản phẩm và danh mục, phục vụ phân tích doanh thu theo ngành hàng và hành vi mua sắm
Columns:
- product_id (VARCHAR(50)): Mã định danh duy nhất của sản phẩm (Unique identifier of the product) PK | constraints: NOT NULL | example: 1e9e8ef04dbcff4541ed26657ea517e5
- product_category_name (VARCHAR(100)): Tên danh mục sản phẩm (bằng tiếng Bồ Đào Nha), có thể được ánh xạ tới product_category_name_translation.product_category_name, nhưng việc ánh xạ có thể không đầy đủ. (Product category name (in Portuguese), can be mapped to product_category_name_translation.product_category_name, but mapping may be incomplete) | allowed values: Refer to 71 category name from file translation (e.g: perfumaria, beleza_saude, esporte_laze, etc.) | constraints: NOT NULL | example: perfumaria
- product_name_lenght (INTEGER): Độ dài tên sản phẩm (số ký tự) (Length of the product name) | constraints: >= 0 | example: 40
- product_description_lenght 

#### KPIs definition

In [ ]:
#import kpi definition

import pandas as pd

# Load table
query = "SELECT * FROM  metadata_kpi_definitions"
df_kpi = pd.read_sql(query, engine)

# Build structured KPI dictionary
kpi_dict = {}

for _, row in df_kpi.iterrows():
    kpi_name = row["kpi_name"]

    # xử lý synonyms (có thể là string "a, b, c" hoặc JSON/list)
    synonyms = row.get("synonyms")
    if isinstance(synonyms, str):
        synonyms_list = [s.strip() for s in synonyms.split(",")]
    else:
        synonyms_list = []

    kpi_dict[kpi_name] = {
        "synonyms": synonyms_list,
        "business_description_en": row.get("business_description_en"),
        "business_description_vi": row.get("business_description_vi"),
        "metric_logic_en": row.get("metric_logic_en"),
        "metric_logic_vi": row.get("metric_logic_vi"),
        "sql_definition": row.get("sql_definition")
    }

# preview
# import json
# print(json.dumps(kpi_dict, indent=2, ensure_ascii=False))

#Format KPI Dictionary

def format_kpi_dict(kpi_dict):
    kpi_text = ""

    for kpi, info in kpi_dict.items():
        kpi_text += f"- KPI Name: {kpi}\n"

        desc_vi = info.get("business_description_vi")
        desc_en = info.get("business_description_en")

        #description = desc_vi or desc_en or "No description"
        metric_logic_vi = info.get("metric_logic_vi")
        metric_logic_en = info.get("metric_logic_en")

        sql_def = info.get("sql_definition")

        synonyms = info.get("synonyms", [])

        if synonyms:
            kpi_text += f"  Synonyms: {', '.join(synonyms)}\n"

        kpi_text += f"  Description: {desc_vi} ({desc_en})\n"
        kpi_text += f"  Metrics Logic: {metric_logic_vi} ({metric_logic_en})\n"
        kpi_text += f"  SQL Logic: {sql_def}\n\n"

    return kpi_text

kpi_text = format_kpi_dict(kpi_dict)
print(kpi_text)


- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SELECT AVG( EXTRACT(EPOCH FROM (order_delivered_customer_date - order_purchase_timestamp)) / 86400.0 ) AS avg_delivery_days FROM orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL;

- KPI Name: Total Orders
  Synonyms: Tổng số đơn hàng
  Description: Tổng số đơn hàng trong hệ thống (h

#### Business Rules

In [ ]:
#import business rules

import pandas as pd

# Load table
query = "SELECT * FROM metadata_business_rules"
df_rules = pd.read_sql(query, engine)

# Build structured business rules dictionary
rules_dict = {}

for _, row in df_rules.iterrows():
    rule_id = row["rule_id"]

    # xử lý applies_to_kpi (có thể là string "kpi1, kpi2")
    applies_to_kpi = row.get("applies_to_kpi")
    if isinstance(applies_to_kpi, str):
        applies_to_kpi_list = [k.strip() for k in applies_to_kpi.split(",")]
    else:
        applies_to_kpi_list = []

    rules_dict[rule_id] = {
        "rule_name": row.get("rule_name"),
        "category": row.get("category"),
        "description": row.get("description"),
        "condition_sql_logic": row.get("condition_sql_logic"),
        "applies_to_kpi": applies_to_kpi_list,
        "is_mandatory": row.get("is_mandatory"),
        "priority": row.get("priority")
    }

# preview
#import json
#print(json.dumps(rules_dict, indent=2, ensure_ascii=False))

#Format Business Rules
def build_rules_prompt(rules_dict):
    rules_by_category = {}

    # Group rules by category
    for rule_id, rule in rules_dict.items():
        category = rule.get("category", "other")
        rules_by_category.setdefault(category, []).append(rule)

    prompt_text = ""

    for category, rules in rules_by_category.items():
        prompt_text += f"### {category.replace('_', ' ').title()} Rules:\n"

        for r in rules:
            desc = r.get("description", "")
            logic = r.get("condition_sql_logic", "")
            mandatory = r.get("is_mandatory", False)

            # 🔥 Convert rule to instruction style
            if "not" in desc.lower() or "không" in desc.lower():
                line = f"- Do NOT: {desc}"
            else:
                line = f"- {desc}"

            # Append logic if useful
            if logic and len(logic) < 120:
                line += f" → {logic}"

            # Mark mandatory rules
            if mandatory:
                line += " (Required)"

            prompt_text += line + "\n"

        prompt_text += "\n"

    return prompt_text

rules_text = build_rules_prompt(rules_dict)
print(rules_text)


### Revenue Rules:
- Chọn đúng nguồn revenue (platform vs seller vs freight) → CASE WHEN question LIKE '%seller%' THEN SUM(order_items.price) ELSE SUM(order_payments.payment_value) END (Required)
- Do NOT: Không được trộn payment_value và price → NOT (use payment_value AND price in same metric) (Required)
- Freight là phí vận chuyển → SUM(freight_value) (Required)

### Time Format Rules:
- Format quý Q1–Q4 → CASE WHEN EXTRACT(MONTH...) THEN 'Qx' END (Required)
- Format tháng chuẩn YYYY-MM → TO_CHAR(timestamp, 'YYYY-MM') (Required)
- Extract năm → EXTRACT(YEAR FROM timestamp) (Required)

### Delivery Rules:
- KPI delivery phải filter delivered (Required)
- 
Tính số ngày giao trễ trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Required)
- Tính số ngày giao trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Required)
- KPI late delivery phải filter delivered (Required)

### Anti Pattern Rules:
- Do NOT: Không dùng COUNT(order_id) → NOT use COUNT(order_id) (Required)
- Do NOT: Không dùn

## 5. Evaluation Utilities
Tạo metrics để đo lường đồng bộ baseline & proposed model

*   Valid SQL Rate (%): Tỷ lệ câu lệnh SQL chạy được trên database mà không báo lỗi cú pháp.
*   Execution Accuracy (EX) (%): Tỷ lệ kết quả truy vấn của Chatbot trùng khớp với kết quả chuẩn (Ground Truth).



In [ ]:
from sqlalchemy import text
import pandas as pd

# ---- Hàm phụ trợ: Chuẩn hóa kết quả SQL ----
def normalize_sql_result(result):
    if not result:
        return []

    normalized_rows = []
    for row in result:
        # Sắp xếp giá trị bên trong từng dòng (xử lý lệch cột)
        vals = list(row)
        sorted_vals = tuple(sorted(vals, key=lambda x: str(x)))
        normalized_rows.append(sorted_vals)

    # Sắp xếp các dòng với nhau (xử lý lệch thứ tự dòng)
    return sorted(normalized_rows, key=lambda row: str(row))


# ---- Metric 1: Valid SQL Rate ----
def valid_sql_rate(df, engine, pred_col):
    success_count = 0
    total = len(df)

    with engine.connect() as conn:
        conn.execute(text("SET search_path TO public"))

        for sql in df[pred_col]:
            try:
                conn.execute(text(sql))
                success_count += 1
            except Exception:
                conn.rollback()  # 👈 quan trọng
                continue

    return success_count / total


# ---- Metric 2: Execution Accuracy ----
def execution_accuracy(df, engine, pred_col, verbose=True):
    correct = 0
    total = len(df)

    with engine.connect() as conn:
        for idx, row in df.iterrows():
            gt_sql = row["ground_truth_sql"]
            pred_sql = row[pred_col]

            try:
                gt_result = conn.execute(text(gt_sql)).fetchall()
                pred_result = conn.execute(text(pred_sql)).fetchall()

                # Thay vì so sánh trực tiếp, ta gọi hàm chuẩn hóa ở đây
                gt_norm = normalize_sql_result(gt_result)
                pred_norm = normalize_sql_result(pred_result)

                if gt_norm == pred_norm:
                    correct += 1
                else:
                    if verbose:
                        print(f"\n[❌ LỆCH KẾT QUẢ] - Index: {idx}")
                        # Nếu trong dataframe có cột 'question' (câu hỏi tự nhiên), in ra luôn cho dễ theo dõi
                        if "question_vie" in df.columns:
                            print(f"Question: {row['question_vie']}")

                        print(f"👉 Ground Truth SQL:\n{gt_sql}")
                        print(f"✅ GT Result (5 rows max): {gt_result[:5]}")

                        print(f"\n👉 Predicted SQL:\n{pred_sql}")
                        print(f"⚠️ Pred Result (5 rows max): {pred_result[:5]}")
                        print("-" * 50)

            except Exception as e:
                if verbose:
                    print(f"\n[🚨 LỖI THỰC THI (Crash)] - Index: {idx}")
                    if "question_vie" in df.columns:
                        print(f"Question: {row['question_vie']}")
                    print(f"👉 Predicted SQL:\n{pred_sql}")
                    print(f"⚠️ Chi tiết lỗi: {str(e).splitlines()[0]}") # In dòng lỗi đầu tiên cho đỡ dài
                    print("-" * 50)

                conn.rollback()  # 👈 quan trọng
                continue

    return correct / total


# ---- Format metrics ----
def pretty_metrics(metrics: dict) -> pd.DataFrame:
    return pd.DataFrame([metrics]).T.rename(columns={0: "value"})


# ---- Evaluation ----
def evaluate(df, engine, pred_col="predicted_sql", verbose=True):

    metrics = {
        "Valid SQL Rate (%)": round(valid_sql_rate(df, engine, pred_col) * 100, 2),
        "Execution Accuracy (%)": round(execution_accuracy(df, engine, pred_col, verbose) * 100, 2)
    }

    return pretty_metrics(metrics)

## 6. Baselines (ROUND 1)
Baselines sử dụng: Few-shot vs. raw schema



#### Set-up API Key

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def clean_sql(sql):
    sql = sql.replace("```sql", "").replace("```", "")
    return sql.strip()



#### Zero-shot vs. raw schema

##### Build-up

In [ ]:
def build_zero_shot_raw_schema_prompt(question, raw_schema_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{raw_schema_text}

### 2. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 3. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_zero_shot_raw_schema_prompt("test", raw_schema_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
Table: products
- product_id (VARCHAR)
- product_category_name (VARCHAR)
- product_name_lenght (BIGINT)
- product_description_lenght (BIGINT)
- product_photos_qty (BIGINT)
- product_weight_g (BIGINT)
- product_length_cm (BIGINT)
- product_height_cm (BIGINT)
- product_width_cm (BIGINT)

Table: order_items
- order_id (VARCHAR)
- order_item_id (INTEGER)
- product_id (VARCHAR)
- seller_id (VARCHAR)
- shipping_limit_date (TIMESTAMP)
- price (NUMERIC)
- freight_value (NUMERIC)

Table: orders
- order_id (VARCHAR)
- customer_id (VARCHAR)
- order_status (VARCHAR)
- order_purchase_timestamp (TIMESTAMP)
- order_approved_at (TIMESTAMP)
- order_delivered_carrier_date (TIMESTAMP)
- order_delivered_cus

In [ ]:
def zero_shot_raw_schema(question):
    prompt = build_zero_shot_raw_schema_prompt(question, raw_schema_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_zero_shot_raw_schema"] = val_df["question_vie"].apply(zero_shot_raw_schema)

# ---- Tính metrics trên validation set ----
results_val_zero_shot_raw_schema = evaluate(val_df, engine, pred_col="predicted_sql_zero_shot_raw_schema")

print(results_val_zero_shot_raw_schema)


[❌ LỆCH KẾT QUẢ] - Index: 0
Question: Bang nào có ít người bán nhất?
👉 Ground Truth SQL:
SELECT seller_state, COUNT(DISTINCT seller_id) AS total_sellers FROM sellers GROUP BY seller_state ORDER BY total_sellers ASC LIMIT 1;
✅ GT Result (5 rows max): [('AC', 1)]

👉 Predicted SQL:
SELECT s.seller_city, COUNT(s.seller_id) AS seller_count
FROM sellers s
GROUP BY s.seller_city
ORDER BY seller_count ASC
LIMIT 1;
⚠️ Pred Result (5 rows max): [('monteiro lobato', 1)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 2
Question: Đếm số lượng bang chỉ có đúng 1 người bán?
👉 Ground Truth SQL:
SELECT COUNT(*) AS total_states FROM (SELECT seller_state FROM sellers GROUP BY seller_state HAVING COUNT(DISTINCT seller_id) = 1) t;
✅ GT Result (5 rows max): [(5,)]

👉 Predicted SQL:
SELECT COUNT(DISTINCT s.seller_state) 
FROM sellers s 
GROUP BY s.seller_state 
HAVING COUNT(s.seller_id) = 1;
⚠️ Pred Result (5 rows max): [(1,), (1,), (1,), (1,), (1,)]
--------------------------

## 7. Proposed Method (ROUND 1)
Proposed model sử dụng: Enhanced Schema Representation (with data dictionary, KPI, simple business rule) & Few-shot Prompting + Detail Guidance for Prompt + Recheck



#### Zero-shot vs. enhanced schema

##### Build-up

In [ ]:
def build_zero_shot_enhanced_schema_prompt(question, enhanced_schema_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 2. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 3. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_zero_shot_enhanced_schema_prompt("test", enhanced_schema_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
### Table: products
Table Description: chứa thông tin về đặc điểm sản phẩm và danh mục, phục vụ phân tích doanh thu theo ngành hàng và hành vi mua sắm
Columns:
- product_id (VARCHAR(50)): Mã định danh duy nhất của sản phẩm (Unique identifier of the product) PK | constraints: NOT NULL | example: 1e9e8ef04dbcff4541ed26657ea517e5
- product_category_name (VARCHAR(100)): Tên danh mục sản phẩm (bằng tiếng Bồ Đào Nha), có thể được ánh xạ tới product_category_name_translation.product_category_name, nhưng việc ánh xạ có thể không đầy đủ. (Product category name (in Portuguese), can be mapped to product_category_name_translation.product_category_name, but mapping may be incomplete) | allowed values

In [ ]:
def zero_shot_enhanced_schema(question):
    prompt = build_zero_shot_enhanced_schema_prompt(question, enhanced_schema_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_zero_shot_enhanced_schema"] = val_df["question_vie"].apply(zero_shot_enhanced_schema)

# ---- Tính metrics trên validation set ----
results_val_zero_shot_enhanced_schema = evaluate(val_df, engine, pred_col="predicted_sql_zero_shot_enhanced_schema")

print(results_val_zero_shot_enhanced_schema)


[❌ LỆCH KẾT QUẢ] - Index: 0
Question: Bang nào có ít người bán nhất?
👉 Ground Truth SQL:
SELECT seller_state, COUNT(DISTINCT seller_id) AS total_sellers FROM sellers GROUP BY seller_state ORDER BY total_sellers ASC LIMIT 1;
✅ GT Result (5 rows max): [('AC', 1)]

👉 Predicted SQL:
SELECT s.seller_state, COUNT(s.seller_id) AS seller_count
FROM sellers s
GROUP BY s.seller_state
ORDER BY seller_count ASC
LIMIT 1;
⚠️ Pred Result (5 rows max): [('AM', 1)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 2
Question: Đếm số lượng bang chỉ có đúng 1 người bán?
👉 Ground Truth SQL:
SELECT COUNT(*) AS total_states FROM (SELECT seller_state FROM sellers GROUP BY seller_state HAVING COUNT(DISTINCT seller_id) = 1) t;
✅ GT Result (5 rows max): [(5,)]

👉 Predicted SQL:
SELECT COUNT(DISTINCT s.seller_state) 
FROM sellers s 
GROUP BY s.seller_state 
HAVING COUNT(s.seller_id) = 1;
⚠️ Pred Result (5 rows max): [(1,), (1,), (1,), (1,), (1,)]
-------------------------------------

#### Zero-shot vs. KPIs

##### Build-up

In [ ]:
def build_zero_shot_kpi_prompt(question, kpi_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

2. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 3. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_zero_shot_kpi_prompt("test", kpi_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def zero_shot_kpi(question):
    prompt = build_zero_shot_kpi_prompt(question, kpi_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_zero_shot_kpi"] = val_df["question_vie"].apply(zero_shot_kpi)

# ---- Tính metrics trên validation set ----
results_val_zero_shot_kpi = evaluate(val_df, engine, pred_col="predicted_sql_zero_shot_kpi")

print(results_val_zero_shot_kpi)


[❌ LỆCH KẾT QUẢ] - Index: 0
Question: Bang nào có ít người bán nhất?
👉 Ground Truth SQL:
SELECT seller_state, COUNT(DISTINCT seller_id) AS total_sellers FROM sellers GROUP BY seller_state ORDER BY total_sellers ASC LIMIT 1;
✅ GT Result (5 rows max): [('AC', 1)]

👉 Predicted SQL:
SELECT t.product_category_name_english AS category, COUNT(DISTINCT oi.seller_id) AS seller_count FROM order_items oi JOIN orders o ON oi.order_id = o.order_id JOIN products pr ON oi.product_id = pr.product_id JOIN product_category_name_translation t ON pr.product_category_name = t.product_category_name GROUP BY t.product_category_name_english ORDER BY seller_count ASC LIMIT 1;
⚠️ Pred Result (5 rows max): [('cds_dvds_musicals', 1)]
--------------------------------------------------

[🚨 LỖI THỰC THI (Crash)] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Predicted SQL:
ERROR: column not found
⚠️ Chi tiết lỗi: (psycopg2.errors.SyntaxError) syntax error at 

#### Zero-shot vs. business Rules

##### Build-up

In [ ]:
def build_zero_shot_rules_prompt(question, rules_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

2. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 3. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_zero_shot_rules_prompt("test", rules_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
### Revenue Rules:
- Chọn đúng nguồn revenue (platform vs seller vs freight) → CASE WHEN question LIKE '%seller%' THEN SUM(order_items.price) ELSE SUM(order_payments.payment_value) END (Required)
- Do NOT: Không được trộn payment_value và price → NOT (use payment_value AND price in same metric) (Required)
- Freight là phí vận chuyển → SUM(freight_value) (Required)

### Time Format Rules:
- Format quý Q1–Q4 → CASE WHEN EXTRACT(MONTH...) THEN 'Qx' END (Required)
- Format tháng chuẩn YYYY-MM → TO_CHAR(timestamp, 'YYYY-MM') (Required)
- Extract năm → EXTRACT(YEAR FROM timestamp) (Required)

### Delivery Rules:
- KPI delivery phải filter delivered (Required)
- 
Tính số ngày giao trễ trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Re

In [ ]:
def zero_shot_rules(question):
    prompt = build_zero_shot_rules_prompt(question, rules_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_zero_shot_rules"] = val_df["question_vie"].apply(zero_shot_rules)

# ---- Tính metrics trên validation set ----
results_val_zero_shot_rules = evaluate(val_df, engine, pred_col="predicted_sql_zero_shot_rules")

print(results_val_zero_shot_rules)


[🚨 LỖI THỰC THI (Crash)] - Index: 0
Question: Bang nào có ít người bán nhất?
👉 Predicted SQL:
ERROR: column not found
⚠️ Chi tiết lỗi: (psycopg2.errors.SyntaxError) syntax error at or near "ERROR"
--------------------------------------------------

[🚨 LỖI THỰC THI (Crash)] - Index: 2
Question: Đếm số lượng bang chỉ có đúng 1 người bán?
👉 Predicted SQL:
ERROR: column not found
⚠️ Chi tiết lỗi: (psycopg2.errors.SyntaxError) syntax error at or near "ERROR"
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 62
Question: Tổng doanh thu của tất cả các người bán trong hệ thống cộng lại là bao nhiêu?
👉 Ground Truth SQL:
SELECT ROUND(SUM(oi.price), 2) AS total_platform_seller_revenue FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered';
✅ GT Result (5 rows max): [(Decimal('13221498.11'),)]

👉 Predicted SQL:
SELECT ROUND(SUM(o.price), 2) AS total_revenue
FROM order_items o;
⚠️ Pred Result (5 rows max): [(Decimal('13591643.70

#### Few-shot vs. raw schema

##### Build-up

In [ ]:
def build_few_shot_raw_schema_prompt(question, raw_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{raw_schema_text}

### 2. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 3. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 4. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_raw_schema_prompt("test", raw_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
Table: products
- product_id (VARCHAR)
- product_category_name (VARCHAR)
- product_name_lenght (BIGINT)
- product_description_lenght (BIGINT)
- product_photos_qty (BIGINT)
- product_weight_g (BIGINT)
- product_length_cm (BIGINT)
- product_height_cm (BIGINT)
- product_width_cm (BIGINT)

Table: order_items
- order_id (VARCHAR)
- order_item_id (INTEGER)
- product_id (VARCHAR)
- seller_id (VARCHAR)
- shipping_limit_date (TIMESTAMP)
- price (NUMERIC)
- freight_value (NUMERIC)

Table: orders
- order_id (VARCHAR)
- customer_id (VARCHAR)
- order_status (VARCHAR)
- order_purchase_timestamp (TIMESTAMP)
- order_approved_at (TIMESTAMP)
- order_delivered_carrier_date (TIMESTAMP)
- order_delivered_cus

In [ ]:
def few_shot_raw_schema(question):
    prompt = build_few_shot_raw_schema_prompt(question, raw_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_raw_schema"] = val_df["question_vie"].apply(few_shot_raw_schema)

# ---- Tính metrics trên validation set ----
results_val_few_shot_raw_schema = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_raw_schema")

print(results_val_few_shot_raw_schema)


[❌ LỆCH KẾT QUẢ] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Ground Truth SQL:
SELECT ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS aov FROM orders o JOIN order_payments p ON o.order_id = p.order_id WHERE o.order_status = 'canceled';
✅ GT Result (5 rows max): [(Decimal('229.21'),)]

👉 Predicted SQL:
SELECT ROUND(AVG(p.payment_value), 2) AS avg_order_value FROM orders o JOIN order_payments p ON o.order_id = p.order_id WHERE o.order_status = 'canceled';
⚠️ Pred Result (5 rows max): [(Decimal('215.75'),)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_ti

#### Few-shot vs. enhanced schema

##### Build-up

In [ ]:
def build_few_shot_enhanced_schema_prompt(question, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 2. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 3. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 4. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_enhanced_schema_prompt("test", enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
### Table: products
Table Description: chứa thông tin về đặc điểm sản phẩm và danh mục, phục vụ phân tích doanh thu theo ngành hàng và hành vi mua sắm
Columns:
- product_id (VARCHAR(50)): Mã định danh duy nhất của sản phẩm (Unique identifier of the product) PK | constraints: NOT NULL | example: 1e9e8ef04dbcff4541ed26657ea517e5
- product_category_name (VARCHAR(100)): Tên danh mục sản phẩm (bằng tiếng Bồ Đào Nha), có thể được ánh xạ tới product_category_name_translation.product_category_name, nhưng việc ánh xạ có thể không đầy đủ. (Product category name (in Portuguese), can be mapped to product_category_name_translation.product_category_name, but mapping may be incomplete) | allowed values

In [ ]:
def few_shot_enhanced_schema(question):
    prompt = build_few_shot_enhanced_schema_prompt(question, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_enhanced_schema"] = val_df["question_vie"].apply(few_shot_enhanced_schema)

# ---- Tính metrics trên validation set ----
results_val_few_shot_enhanced_schema = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_enhanced_schema")

print(results_val_few_shot_enhanced_schema)


[❌ LỆCH KẾT QUẢ] - Index: 62
Question: Tổng doanh thu của tất cả các người bán trong hệ thống cộng lại là bao nhiêu?
👉 Ground Truth SQL:
SELECT ROUND(SUM(oi.price), 2) AS total_platform_seller_revenue FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered';
✅ GT Result (5 rows max): [(Decimal('13221498.11'),)]

👉 Predicted SQL:
SELECT ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered';
⚠️ Pred Result (5 rows max): [(Decimal('15419773.75'),)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Ground Truth SQL:
SELECT ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS aov FROM orders o JOIN order_payments p ON o.order_id = p.order_id WHERE o.order_status = 'canceled';
✅ GT Result (5 rows max): [(Decimal('229.21

#### Few-shot vs. KPIs

##### Build-up

In [ ]:
def build_few_shot_kpi_prompt(question, kpi_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 2. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 3. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 4. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_kpi_prompt("test", kpi_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def few_shot_kpi(question):
    prompt = build_few_shot_kpi_prompt(question, kpi_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_kpi"] = val_df["question_vie"].apply(few_shot_kpi)

# ---- Tính metrics trên validation set ----
results_val_few_shot_kpi = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_kpi")

print(results_val_few_shot_kpi)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH seller_orders AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS order_count FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND o.order_purchase_timestamp >= '2017-10-01 00:00:00' AND o.order_purchase_timestamp < '2018-01-01 00:00:00' GROUP BY oi

#### Few-shot vs. business rules

##### Build-up

In [ ]:
def build_few_shot_rules_prompt(question, rules_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 2. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 3. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 4. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_rules_prompt("test", rules_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
### Revenue Rules:
- Chọn đúng nguồn revenue (platform vs seller vs freight) → CASE WHEN question LIKE '%seller%' THEN SUM(order_items.price) ELSE SUM(order_payments.payment_value) END (Required)
- Do NOT: Không được trộn payment_value và price → NOT (use payment_value AND price in same metric) (Required)
- Freight là phí vận chuyển → SUM(freight_value) (Required)

### Time Format Rules:
- Format quý Q1–Q4 → CASE WHEN EXTRACT(MONTH...) THEN 'Qx' END (Required)
- Format tháng chuẩn YYYY-MM → TO_CHAR(timestamp, 'YYYY-MM') (Required)
- Extract năm → EXTRACT(YEAR FROM timestamp) (Required)

### Delivery Rules:
- KPI delivery phải filter delivered (Required)
- 
Tính số ngày giao trễ trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Re

In [ ]:
def few_shot_rules(question):
    prompt = build_few_shot_rules_prompt(question, rules_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_rules"] = val_df["question_vie"].apply(few_shot_rules)

# ---- Tính metrics trên validation set ----
results_val_few_shot_rules = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_rules")

print(results_val_few_shot_rules)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH base AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS order_count FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND EXTRACT(MONTH FROM o.order_purchase_timestamp) IN (10, 11, 12) AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 GROUP B

## 8. Model Comparision (ROUND 1)
Tạo bảng so sánh baseline vs proposed trên validation.

### Val Result Comparision

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Baseline - Zero-shot vs. raw schema", "Proposed - Zero-shot vs. enhanced schema", "Proposed - Zero-shot vs. KPIs", "Proposed - Zero-shot vs. business rules", "Proposed - Few-shot vs. raw schema", "Proposed - Few-shot vs. enhanced schema", "Proposed - Few-shot vs. KPIs", "Proposed - Few-shot vs. business rules"],
    "Valid SQL Rate (%)": [
         results_val_zero_shot_raw_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_zero_shot_enhanced_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_zero_shot_kpi.loc["Valid SQL Rate (%)", "value"],
         results_val_zero_shot_rules.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_raw_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_enhanced_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_kpi.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_rules.loc["Valid SQL Rate (%)", "value"]
    ],
    "Execution Accuracy (%)": [
        results_val_zero_shot_raw_schema.loc["Execution Accuracy (%)", "value"],
         results_val_zero_shot_enhanced_schema.loc["Execution Accuracy (%)", "value"],
         results_val_zero_shot_kpi.loc["Execution Accuracy (%)", "value"],
         results_val_zero_shot_rules.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_raw_schema.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_enhanced_schema.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_kpi.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_rules.loc["Execution Accuracy (%)", "value"]
    ]
})

print(comparison.to_markdown(index=False, floatfmt=".2f"))

| Model                                    |   Valid SQL Rate (%) |   Execution Accuracy (%) |
|:-----------------------------------------|---------------------:|-------------------------:|
| Baseline - Zero-shot vs. raw schema      |                88.89 |                     0.00 |
| Proposed - Zero-shot vs. enhanced schema |               100.00 |                     5.56 |
| Proposed - Zero-shot vs. KPIs            |                66.67 |                    22.22 |
| Proposed - Zero-shot vs. business rules  |                16.67 |                     5.56 |
| Proposed - Few-shot vs. raw schema       |                88.89 |                    33.33 |
| Proposed - Few-shot vs. enhanced schema  |                88.89 |                    27.78 |
| Proposed - Few-shot vs. KPIs             |                88.89 |                    38.89 |
| Proposed - Few-shot vs. business rules   |                83.33 |                    44.44 |


#### Some insights
##### - KPIs là loại ngữ cảnh (Context) mang lại hiệu quả cao (cứu được zero-shot rất mạnh)
##### - Business rules là loại ngữ cảnh (context) chỉ hiệu quả khi có examples đi kèm
##### - Enhanced Schema chưa chứng minh được lợi ích rõ rệt (lúc tăng độ chính xác, lúc lại giảm độ chính xác)
##### --> Dùng (Few-shot + Business Rules) và (Few-shot + KPIs) làm base tiếp theo | thử (Few-shot + KPIs + Business Rules) | (Few-shot + KPIs + Business Rules + Raw Schema) | (Few-shot + KPIs + Business Rules + Enhanced Schema)

## 9. Proposed Method (ROUND 2)



#### Few-shot vs. KPIs vs. business rules

##### Build-up

In [ ]:
def build_few_shot_kpi_rules_prompt(question, kpi_text, rules_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 2. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 3. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 4. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_kpi_rules_prompt("test", kpi_text, rules_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def few_shot_kpi_rules(question):
    prompt = build_few_shot_kpi_rules_prompt(question, kpi_text, rules_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_kpi_rules"] = val_df["question_vie"].apply(few_shot_kpi_rules)

# ---- Tính metrics trên validation set ----
results_val_few_shot_kpi_rules = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_kpi_rules")

print(results_val_few_shot_kpi_rules)


[🚨 LỖI THỰC THI (Crash)] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Predicted SQL:
SELECT ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS aov FROM order_payments p JOIN orders o ON p.order_id = o.order_id WHERE o.order_status = 'canceled';
⚠️ Chi tiết lỗi: (psycopg2.errors.AmbiguousColumn) column reference "order_id" is ambiguous
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 49

#### Few-shot vs. KPIs vs. business rules vs. raw schema

##### Build-up

In [ ]:
def build_few_shot_kpi_rules_raw_schema_prompt(question, kpi_text, rules_text, raw_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 2. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 3. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{raw_schema_text}

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_kpi_rules_raw_schema_prompt("test", kpi_text, rules_text, raw_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def few_shot_kpi_rules_raw_schema(question):
    prompt = build_few_shot_kpi_rules_raw_schema_prompt(question, kpi_text, rules_text, raw_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_kpi_rules_raw_schema"] = val_df["question_vie"].apply(few_shot_kpi_rules_raw_schema)

# ---- Tính metrics trên validation set ----
results_val_few_shot_kpi_rules_raw_schema = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_kpi_rules_raw_schema")

print(results_val_few_shot_kpi_rules_raw_schema)


[🚨 LỖI THỰC THI (Crash)] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Predicted SQL:
ERROR: column not found
⚠️ Chi tiết lỗi: (psycopg2.errors.SyntaxError) syntax error at or near "ERROR"
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH b

#### Few-shot vs. KPIs vs. business rules vs. enhanced schema

##### Build-up

In [ ]:
def build_few_shot_kpi_rules_enhanced_schema_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 2. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 3. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_kpi_rules_enhanced_schema_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def few_shot_kpi_rules_enhanced_schema(question):
    prompt = build_few_shot_kpi_rules_enhanced_schema_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_kpi_rules_enhanced_schema"] = val_df["question_vie"].apply(few_shot_kpi_rules_enhanced_schema)

# ---- Tính metrics trên validation set ----
results_val_few_shot_kpi_rules_enhanced_schema = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_kpi_rules_enhanced_schema")

print(results_val_few_shot_kpi_rules_enhanced_schema)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH base AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS total_orders FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(MONTH FROM o.order_purchase_timestamp) IN (10, 11, 12) GROUP 

## 10. Model Comparision (ROUND 2)
Tạo bảng so sánh baseline vs proposed trên validation.

### Val Result Comparision

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Baseline - Few-shot vs. KPIs", "Baseline - Few-shot vs. business rules", "Proposed - Few-shot vs. KPIs vs. business rules", "Proposed - Few-shot vs. KPIs vs. business rules vs. raw schema", "Proposed - Few-shot vs. KPIs vs. business rules vs. enhanced schema"],
    "Valid SQL Rate (%)": [
         results_val_few_shot_kpi.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_rules.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_kpi_rules.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_kpi_rules_raw_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_kpi_rules_enhanced_schema.loc["Valid SQL Rate (%)", "value"]
    ],
    "Execution Accuracy (%)": [
        results_val_few_shot_kpi.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_rules.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_kpi_rules.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_kpi_rules_raw_schema.loc["Execution Accuracy (%)", "value"],
        results_val_few_shot_kpi_rules_enhanced_schema.loc["Execution Accuracy (%)", "value"]
    ]
})

print(comparison.to_markdown(index=False, floatfmt=".2f"))

| Model                                                               |   Valid SQL Rate (%) |   Execution Accuracy (%) |
|:--------------------------------------------------------------------|---------------------:|-------------------------:|
| Baseline - Few-shot vs. KPIs                                        |                88.89 |                    38.89 |
| Baseline - Few-shot vs. business rules                              |                83.33 |                    44.44 |
| Proposed - Few-shot vs. KPIs vs. business rules                     |                77.78 |                    44.44 |
| Proposed - Few-shot vs. KPIs vs. business rules vs. raw schema      |                88.89 |                    50.00 |
| Proposed - Few-shot vs. KPIs vs. business rules vs. enhanced schema |                88.89 |                    55.56 |


#### Some insights
##### - Context không cộng dồn tuyến tính → KPI + Rules không tự tăng hiệu quả nếu không có schema hỗ trợ.
##### - Best combo rõ ràng: Few-shot + KPI + business rules + enhanced schema → 55.56% (highest) --> model nên chọn
##### - Thử tiếp các thứ tự khác nhau của bộ 3 (KPI + business rules + enhanced schema) | giữ nguyên few-shot examples làm foundation


## 11. Proposed Method (ROUND 3)



#### Few-shot vs. KPIs vs. enhanced schema vs. business rules

##### Build-up

In [ ]:
def build_few_shot_kpi_enhanced_schema_rules_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 2. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 3. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_kpi_enhanced_schema_rules_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
- KPI Name: Total Sellers
  Synonyms: Tổng số người bán, Tổng số seller
  Description: Tổng số người bán đã đăng ký trên hệ thống. (Number of registered sellers.)
  Metrics Logic: Đếm số lượng ID người bán duy nhất (Count of distinct seller IDs)
  SQL Logic: SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;

- KPI Name: Delivery Time (Average)
  Synonyms: Thời gian giao hàng trung bình
  Description: Thời gian giao hàng trung bình từ lúc khách đặt đến khi nhận hàng. (Average time taken from order placement to customer delivery.)
  Metrics Logic: Tổng thời gian giao hàng / Số đơn (Number of deliveries divided by total number of delivered orders)
  SQL Logic: SEL

In [ ]:
def few_shot_kpi_enhanced_schema_rules(question):
    prompt = build_few_shot_kpi_enhanced_schema_rules_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_kpi_enhanced_schema_rules"] = val_df["question_vie"].apply(few_shot_kpi_enhanced_schema_rules)

# ---- Tính metrics trên validation set ----
results_val_few_shot_kpi_enhanced_schema_rules = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_kpi_enhanced_schema_rules")

print(results_val_few_shot_kpi_enhanced_schema_rules)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH base AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS total_orders FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND EXTRACT(MONTH FROM o.order_purchase_timestamp) IN (10, 11, 12) AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 GROUP 

#### Few-shot vs. business rules vs. KPIs vs. enhanced schema

##### Build-up

In [ ]:
def build_few_shot_rules_kpi_enhanced_schema_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 2. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 3. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_rules_kpi_enhanced_schema_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
### Revenue Rules:
- Chọn đúng nguồn revenue (platform vs seller vs freight) → CASE WHEN question LIKE '%seller%' THEN SUM(order_items.price) ELSE SUM(order_payments.payment_value) END (Required)
- Do NOT: Không được trộn payment_value và price → NOT (use payment_value AND price in same metric) (Required)
- Freight là phí vận chuyển → SUM(freight_value) (Required)

### Time Format Rules:
- Format quý Q1–Q4 → CASE WHEN EXTRACT(MONTH...) THEN 'Qx' END (Required)
- Format tháng chuẩn YYYY-MM → TO_CHAR(timestamp, 'YYYY-MM') (Required)
- Extract năm → EXTRACT(YEAR FROM timestamp) (Required)

### Delivery Rules:
- KPI delivery phải filter delivered (Required)
- 
Tính số ngày giao trễ trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Re

In [ ]:
def few_shot_rules_kpi_enhanced_schema(question):
    prompt = build_few_shot_rules_kpi_enhanced_schema_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_rules_kpi_enhanced_schema"] = val_df["question_vie"].apply(few_shot_rules_kpi_enhanced_schema)

# ---- Tính metrics trên validation set ----
results_val_few_shot_rules_kpi_enhanced_schema = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_rules_kpi_enhanced_schema")

print(results_val_few_shot_rules_kpi_enhanced_schema)


[🚨 LỖI THỰC THI (Crash)] - Index: 20
Question: giá trị đơn hàng trung bình của các đơn hàng đã bị hủy (canceled) là bao nhiêu?
👉 Predicted SQL:
ERROR: column not found
⚠️ Chi tiết lỗi: (psycopg2.errors.SyntaxError) syntax error at or near "ERROR"
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH b

#### Few-shot vs. business rules vs. enhanced schema vs. KPIs

##### Build-up

In [ ]:
def build_few_shot_rules_enhanced_schema_kpi_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 2. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 3. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_rules_enhanced_schema_kpi_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
### Revenue Rules:
- Chọn đúng nguồn revenue (platform vs seller vs freight) → CASE WHEN question LIKE '%seller%' THEN SUM(order_items.price) ELSE SUM(order_payments.payment_value) END (Required)
- Do NOT: Không được trộn payment_value và price → NOT (use payment_value AND price in same metric) (Required)
- Freight là phí vận chuyển → SUM(freight_value) (Required)

### Time Format Rules:
- Format quý Q1–Q4 → CASE WHEN EXTRACT(MONTH...) THEN 'Qx' END (Required)
- Format tháng chuẩn YYYY-MM → TO_CHAR(timestamp, 'YYYY-MM') (Required)
- Extract năm → EXTRACT(YEAR FROM timestamp) (Required)

### Delivery Rules:
- KPI delivery phải filter delivered (Required)
- 
Tính số ngày giao trễ trung bình → AVG(EXTRACT(EPOCH FROM (...))/86400) (Re

In [ ]:
def few_shot_rules_enhanced_schema_kpi(question):
    prompt = build_few_shot_rules_enhanced_schema_kpi_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_rules_enhanced_schema_kpi"] = val_df["question_vie"].apply(few_shot_rules_enhanced_schema_kpi)

# ---- Tính metrics trên validation set ----
results_val_few_shot_rules_enhanced_schema_kpi = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_rules_enhanced_schema_kpi")

print(results_val_few_shot_rules_enhanced_schema_kpi)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH base AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS order_count FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND EXTRACT(MONTH FROM o.order_purchase_timestamp) IN (10, 11, 12) AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 GROUP B

#### Few-shot vs. enhanced schema vs. KPIs vs. business rules

##### Build-up

In [ ]:
def build_few_shot_enhanced_schema_kpi_rules_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 2. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 3. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_enhanced_schema_kpi_rules_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
### Table: products
Table Description: chứa thông tin về đặc điểm sản phẩm và danh mục, phục vụ phân tích doanh thu theo ngành hàng và hành vi mua sắm
Columns:
- product_id (VARCHAR(50)): Mã định danh duy nhất của sản phẩm (Unique identifier of the product) PK | constraints: NOT NULL | example: 1e9e8ef04dbcff4541ed26657ea517e5
- product_category_name (VARCHAR(100)): Tên danh mục sản phẩm (bằng tiếng Bồ Đào Nha), có thể được ánh xạ tới product_category_name_translation.product_category_name, nhưng việc ánh xạ có thể không đầy đủ. (Product category name (in Portuguese), can be mapped to product_category_name_translation.product_category_name, but mapping may be incomplete) | allowed values

In [ ]:
def few_shot_enhanced_schema_kpi_rules(question):
    prompt = build_few_shot_enhanced_schema_kpi_rules_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_enhanced_schema_kpi_rules"] = val_df["question_vie"].apply(few_shot_enhanced_schema_kpi_rules)

# ---- Tính metrics trên validation set ----
results_val_few_shot_enhanced_schema_kpi_rules = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_enhanced_schema_kpi_rules")

print(results_val_few_shot_enhanced_schema_kpi_rules)


[❌ LỆCH KẾT QUẢ] - Index: 62
Question: Tổng doanh thu của tất cả các người bán trong hệ thống cộng lại là bao nhiêu?
👉 Ground Truth SQL:
SELECT ROUND(SUM(oi.price), 2) AS total_platform_seller_revenue FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered';
✅ GT Result (5 rows max): [(Decimal('13221498.11'),)]

👉 Predicted SQL:
SELECT ROUND(SUM(payment_value), 2) AS total_revenue FROM order_payments;
⚠️ Pred Result (5 rows max): [(Decimal('16008872.12'),)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result

#### Few-shot vs. enhanced schema vs. business rules vs. KPIs

##### Build-up

In [ ]:
def build_few_shot_enhanced_schema_rules_kpi_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text):

    prompt = f"""You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
{enhanced_schema_text}

### 2. BUSINESS & SQL RULES
Adhere strictly to the following constraints:
{rules_text}
- Always use proper JOIN conditions based on the schema relationships.
- Use ILIKE for case-insensitive string matching when searching text in Vietnamese.
- Handle NULL values safely (e.g., use COALESCE) in mathematical operations.

### 3. KPI DEFINITIONS & FORMULAS (CRITICAL)
When the user asks about specific metrics, you MUST use the following exact formulas:
{kpi_text}

### 4. EXAMPLES (REFERENCE ONLY)
Study these examples to understand the expected translation logic. DO NOT copy these blindly, adapt the logic to the new question.
{example_text}
### END OF EXAMPLES

### 5. VALIDATION & STRICT CONSTRAINTS (MANDATORY)
You MUST strictly follow these rules:

1. COLUMN EXISTENCE:
- Every column used MUST exist in the schema.
- DO NOT invent, assume, or hallucinate any column.

2. COLUMN OWNERSHIP:
- Each column belongs to EXACTLY ONE table.
- You MUST identify the correct table for every column.
- ALL columns MUST be prefixed with table alias (e.g., s.seller_state).

3. JOIN REQUIREMENT:
- If a column is not in the current table, you MUST find the correct table using foreign key relationships.
- You MUST add the correct JOIN to access that column.
- DO NOT assume the column exists in already joined tables.

4. STRICT FAILURE:
- If a required column does NOT exist in the schema:
  → Return EXACTLY: ERROR: column not found
- DO NOT guess or approximate.
- If the query contains division (/) or AVG() → MUST wrap the result with ROUND(..., 2)
- If ROUND is missing → REWRITE the query before returning

5. FINAL VALIDATION BEFORE OUTPUT:
- Verify every column exists in its referenced table
- Verify all required tables are properly joined
- Verify no column is used without table alias

### 6. FINAL INSTRUCTIONS
- Return ONLY one of the following:
  1. A valid PostgreSQL SQL query
  2. ERROR: column not found
- NO explanations, NO introductory text.
- NO markdown formatting.
- Generate exactly ONE output.

### USER QUESTION
{question}

### SQL:
"""
    return prompt.strip()

In [ ]:
#xem thử prompt
prompt = build_few_shot_enhanced_schema_rules_kpi_prompt("test", kpi_text, rules_text, enhanced_schema_text, example_text)

print(prompt)

You are an elite PostgreSQL Expert and Data Analyst. Your task is to translate Vietnamese natural language questions into highly accurate, optimized, and executable PostgreSQL queries.

### 1. DATABASE SCHEMA
Use ONLY the tables and columns provided below. Pay attention to data types and relationships.
### Table: products
Table Description: chứa thông tin về đặc điểm sản phẩm và danh mục, phục vụ phân tích doanh thu theo ngành hàng và hành vi mua sắm
Columns:
- product_id (VARCHAR(50)): Mã định danh duy nhất của sản phẩm (Unique identifier of the product) PK | constraints: NOT NULL | example: 1e9e8ef04dbcff4541ed26657ea517e5
- product_category_name (VARCHAR(100)): Tên danh mục sản phẩm (bằng tiếng Bồ Đào Nha), có thể được ánh xạ tới product_category_name_translation.product_category_name, nhưng việc ánh xạ có thể không đầy đủ. (Product category name (in Portuguese), can be mapped to product_category_name_translation.product_category_name, but mapping may be incomplete) | allowed values

In [ ]:
def few_shot_enhanced_schema_rules_kpi(question):
    prompt = build_few_shot_enhanced_schema_rules_kpi_prompt(question, kpi_text, rules_text, enhanced_schema_text, example_text)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content
    return clean_sql(sql)

##### Validation

In [ ]:
# ---- Predict SQL cho validation set ----
val_df["predicted_sql_few_shot_enhanced_schema_rules_kpi"] = val_df["question_vie"].apply(few_shot_enhanced_schema_rules_kpi)

# ---- Tính metrics trên validation set ----
results_val_few_shot_enhanced_schema_rules_kpi = evaluate(val_df, engine, pred_col="predicted_sql_few_shot_enhanced_schema_rules_kpi")

print(results_val_few_shot_enhanced_schema_rules_kpi)


[❌ LỆCH KẾT QUẢ] - Index: 69
Question: Top 10 seller có số lượng đơn hàng nhiều nhất trong quý 4 năm 2017
👉 Ground Truth SQL:
SELECT oi.seller_id, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 AND EXTRACT(QUARTER FROM o.order_purchase_timestamp) = 4 GROUP BY oi.seller_id ORDER BY total_orders DESC LIMIT 10;
✅ GT Result (5 rows max): [('cc419e0650a3c5ba77189a1882b7556a', 513), ('1f50f920176fa81dab994f9023523100', 494), ('6560211a19b47992c3666cc44a7e94c0', 382), ('4a3ca9315b744ce9f8e9374361493884', 379), ('3d871de0142ce09b7081e2b9d1733cb1', 318)]

👉 Predicted SQL:
WITH base AS (SELECT oi.seller_id, COUNT(DISTINCT o.order_id) AS order_count FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.order_status = 'delivered' AND EXTRACT(MONTH FROM o.order_purchase_timestamp) IN (10, 11, 12) AND EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 GROUP B

## 12. Model Comparision (ROUND 3)
Tạo bảng so sánh baseline vs proposed trên validation.

### Val Result Comparision

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Baseline - Few-shot vs. KPIs vs. business rules vs. enhanced schema","Proposed - Few-shot vs. KPIs vs. enhanced schema vs. business rules","Proposed - Few-shot vs. business rules vs. KPIs vs. enhanced schema","Proposed - Few-shot vs. business rules vs. enhanced schema vs. KPIs","Proposed - Few-shot vs. enhanced schema vs. KPIs vs. business rules","Proposed - Few-shot vs. enhanced schema vs. business rules vs. KPIs"],
    "Valid SQL Rate (%)": [
         results_val_few_shot_kpi_rules_enhanced_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_kpi_enhanced_schema_rules.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_rules_kpi_enhanced_schema.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_rules_enhanced_schema_kpi.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_enhanced_schema_kpi_rules.loc["Valid SQL Rate (%)", "value"],
         results_val_few_shot_enhanced_schema_rules_kpi.loc["Valid SQL Rate (%)", "value"]
    ],
    "Execution Accuracy (%)": [
         results_val_few_shot_kpi_rules_enhanced_schema.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_kpi_enhanced_schema_rules.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_rules_kpi_enhanced_schema.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_rules_enhanced_schema_kpi.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_enhanced_schema_kpi_rules.loc["Execution Accuracy (%)", "value"],
         results_val_few_shot_enhanced_schema_rules_kpi.loc["Execution Accuracy (%)", "value"]
    ]
})

print(comparison.to_markdown(index=False, floatfmt=".2f"))

| Model                                                               |   Valid SQL Rate (%) |   Execution Accuracy (%) |
|:--------------------------------------------------------------------|---------------------:|-------------------------:|
| Baseline - Few-shot vs. KPIs vs. business rules vs. enhanced schema |                88.89 |                    55.56 |
| Proposed - Few-shot vs. KPIs vs. enhanced schema vs. business rules |                94.44 |                    66.67 |
| Proposed - Few-shot vs. business rules vs. KPIs vs. enhanced schema |                88.89 |                    50.00 |
| Proposed - Few-shot vs. business rules vs. enhanced schema vs. KPIs |                83.33 |                    55.56 |
| Proposed - Few-shot vs. enhanced schema vs. KPIs vs. business rules |               100.00 |                    61.11 |
| Proposed - Few-shot vs. enhanced schema vs. business rules vs. KPIs |                88.89 |                    55.56 |


#### Some insights
##### - Thứ tự của các thành phần context có ảnh hưởng đáng kể đến hiệu suất mô hình (dao động 50% - 66.67%)
##### - Thứ tự tốt nhất là: KPIs > Enhanced Schema > Rules (66.67%, cải thiện 11% về accuracy) và Thứ tự tốt thứ 2 là: Schema > KPIs > Rules (61.11%, cải thiện 6% về accuracy | 100%, cải thiện 12% về valid)
##### -  2 case tốt nhất này đã chỉ ra:
###### + KPIs nên đứng trước Rules (model cần hiểu tính gì trước khi áp constraint (rules)
###### + Rules nên đặt cuối (Rules hoạt động tốt nhất khi refine logic sau khi model đã hiểu KPI + schema)
###### + Schema không nên đặt cuối (bị quên)
##### - Case tệ nhất là Rules > KPIs > Schema (model bị: áp rule trước khi hiểu mục tiêu)
##### --> Lấy 2 cấu hình tiềm năng nhất là (Few-shot vs. KPIs vs. enhanced schema vs. business rules) và (Few-shot vs. enhanced schema vs. KPIs vs. business rules) để thử nghiệm trên tập test

## 13. Final Test Evaluation

### Test
##### Chỉ chạy trên (Baseline - Zero-shot vs. raw schema) | (Baseline - Few-shot vs. KPIs) | (Baseline - Few-shot vs. business rules) | (Baseline - Few-shot vs. KPIs vs. business rules vs. enhanced schema) và (Proposed - Few-shot vs. KPIs vs. enhanced schema vs. business rules) | (Proposed - Few-shot vs. enhanced schema vs. KPIs vs. business rules)

In [ ]:

# ---- Predict SQL cho test set ----
test_df["predicted_sql_zero_shot_raw_schema"] = test_df["question_vie"].apply(zero_shot_raw_schema)

# ---- Tính metrics trên test set ----
results_test_zero_shot_raw_schema = evaluate(test_df, engine, pred_col="predicted_sql_zero_shot_raw_schema")

#===============================================================

# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_kpi"] = test_df["question_vie"].apply(few_shot_kpi)

# ---- Tính metrics trên test set ----
results_test_few_shot_kpi = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_kpi")

#===============================================================

# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_rules"] = test_df["question_vie"].apply(few_shot_rules)

# ---- Tính metrics trên test set ----
results_test_few_shot_rules = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_rules")

#===============================================================

# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_kpi_rules_enhanced_schema"] = test_df["question_vie"].apply(few_shot_kpi_rules_enhanced_schema)

# ---- Tính metrics trên test set ----
results_test_few_shot_kpi_rules_enhanced_schema = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_kpi_rules_enhanced_schema")

#===============================================================

# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_kpi_enhanced_schema_rules"] = test_df["question_vie"].apply(few_shot_kpi_enhanced_schema_rules)

# ---- Tính metrics trên test set ----
results_test_few_shot_kpi_enhanced_schema_rules = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_kpi_enhanced_schema_rules")

#===============================================================

# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_enhanced_schema_kpi_rules"] = test_df["question_vie"].apply(few_shot_enhanced_schema_kpi_rules)

# ---- Tính metrics trên test set ----
results_test_few_shot_enhanced_schema_kpi_rules = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_enhanced_schema_kpi_rules")

print(results_test_few_shot_enhanced_schema_kpi_rules)



[❌ LỆCH KẾT QUẢ] - Index: 23
Question: giá trị đơn hàng trung bình của những đơn hàng mà tổng phí vận chuyển (freight) đắt hơn cả tổng giá trị sản phẩm (price) là bao nhiêu?
👉 Ground Truth SQL:
WITH order_cost_breakdown AS (SELECT order_id, SUM(price) AS total_price, SUM(freight_value) AS total_freight FROM order_items GROUP BY order_id) SELECT ROUND(SUM(p.payment_value) / COUNT(DISTINCT p.order_id), 2) AS aov FROM order_payments p JOIN order_cost_breakdown f ON p.order_id = f.order_id WHERE f.total_freight > f.total_price;
✅ GT Result (5 rows max): [(Decimal('51.21'),)]

👉 Predicted SQL:
SELECT ROUND(AVG(op.payment_value), 2) AS average_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE oi.freight_value > oi.price;
⚠️ Pred Result (5 rows max): [(Decimal('72.16'),)]
--------------------------------------------------

[❌ LỆCH KẾT QUẢ] - Index: 79
Question: Top 5 danh mục sản phẩm có mức tăng số lượng đơn hà

In [ ]:
# ---- Predict SQL cho test set ----
test_df["predicted_sql_few_shot_enhanced_schema_kpi_rules"] = test_df["question_vie"].apply(few_shot_enhanced_schema_kpi_rules)

# ---- Tính metrics trên test set ----
results_test_few_shot_enhanced_schema_kpi_rules = evaluate(test_df, engine, pred_col="predicted_sql_few_shot_enhanced_schema_kpi_rules")

print(results_test_few_shot_enhanced_schema_kpi_rules)


[❌ LỆCH KẾT QUẢ] - Index: 79
Question: Top 5 danh mục sản phẩm có mức tăng số lượng đơn hàng cao nhất khi so sánh giữa nửa đầu năm 2017 (Quý 1–2) và nửa cuối năm 2017 (Quý 3–4)
👉 Ground Truth SQL:
WITH category_half AS ( SELECT t.product_category_name_english, CASE WHEN EXTRACT(QUARTER FROM o.order_purchase_timestamp) IN (1, 2) THEN 'H1' ELSE 'H2' END AS half_year, COUNT(DISTINCT oi.order_id) AS total_orders FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id JOIN products AS p ON oi.product_id = p.product_id JOIN product_category_name_translation AS t ON p.product_category_name=t.product_category_name WHERE EXTRACT(YEAR FROM o.order_purchase_timestamp) = 2017 GROUP BY t.product_category_name_english, CASE WHEN EXTRACT(QUARTER FROM o.order_purchase_timestamp) IN (1, 2) THEN 'H1' ELSE 'H2' END ), pivot AS ( SELECT product_category_name_english, MAX(CASE WHEN half_year = 'H1' THEN total_orders ELSE 0 END) AS orders_h1, MAX(CASE WHEN half_year = 'H2' THEN total_orders ELS

### Test Result Comparision

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Baseline - Zero-shot vs. raw schema","Baseline - Few-shot vs. KPIs", "Baseline - Few-shot vs. business rules","Baseline - Few-shot vs. KPIs vs. business rules vs. enhanced schema","Proposed - Few-shot vs. KPIs vs. enhanced schema vs. business rules","Proposed - Few-shot vs. enhanced schema vs. KPIs vs. business rules"],
    "Valid SQL Rate (%)": [
         results_test_zero_shot_raw_schema.loc["Valid SQL Rate (%)", "value"],
         results_test_few_shot_kpi.loc["Valid SQL Rate (%)", "value"],
         results_test_few_shot_rules.loc["Valid SQL Rate (%)", "value"],
         results_test_few_shot_kpi_rules_enhanced_schema.loc["Valid SQL Rate (%)", "value"],
         results_test_few_shot_kpi_enhanced_schema_rules.loc["Valid SQL Rate (%)", "value"],
         results_test_few_shot_enhanced_schema_kpi_rules.loc["Valid SQL Rate (%)", "value"]
    ],
    "Execution Accuracy (%)": [
         results_test_zero_shot_raw_schema.loc["Execution Accuracy (%)", "value"],
         results_test_few_shot_kpi.loc["Execution Accuracy (%)", "value"],
         results_test_few_shot_rules.loc["Execution Accuracy (%)", "value"],
         results_test_few_shot_kpi_rules_enhanced_schema.loc["Execution Accuracy (%)", "value"],
         results_test_few_shot_kpi_enhanced_schema_rules.loc["Execution Accuracy (%)", "value"],
         results_test_few_shot_enhanced_schema_kpi_rules.loc["Execution Accuracy (%)", "value"]
    ]
})

print(comparison.to_markdown(index=False, floatfmt=".2f"))



| Model                                                               |   Valid SQL Rate (%) |   Execution Accuracy (%) |
|:--------------------------------------------------------------------|---------------------:|-------------------------:|
| Baseline - Zero-shot vs. raw schema                                 |                86.11 |                    22.22 |
| Baseline - Few-shot vs. KPIs                                        |                84.72 |                    48.61 |
| Baseline - Few-shot vs. business rules                              |                86.11 |                    43.06 |
| Baseline - Few-shot vs. KPIs vs. business rules vs. enhanced schema |                95.83 |                    59.72 |
| Proposed - Few-shot vs. KPIs vs. enhanced schema vs. business rules |                98.61 |                    59.72 |
| Proposed - Few-shot vs. enhanced schema vs. KPIs vs. business rules |                94.44 |                    66.67 |


#### Some insights
##### - Mô hình tốt nhất cho bài toán này là sự kết hợp có cấu trúc → Few-shot (pattern) + [ Enhanced Schema (grounding)+  KPIs/ Business Rules (logic) ] = hiệu suất tối ưu cho bài toán text-to-SQL dạng business (+8% valid và +44% accuracy)
##### - Giá trị cốt lõi của Prompt Engineering: Tinh chỉnh prompt giúp độ chính xác thực thi (Execution Accuracy) tăng vọt gấp 3 lần (từ 22.2% lên 66.7%) so với việc chỉ đưa raw schema cho LLM.
##### - Sức mạnh cộng hưởng dữ liệu: Việc cung cấp trọn bộ bối cảnh (Enhaned Schema + KPIs + Rules) giúp LLM hiểu nghiệp vụ sâu hơn hẳn so với việc chỉ đưa từng thành phần đơn lẻ.
##### - Tư duy "Xây móng trước": Khi gặp câu hỏi lạ (không quen thuộc như Val - nên kết quả khác Val), LLM cần đọc Cấu trúc Database (Schema) trước để định hình các bảng/cột, sau đó mới tiếp thu Công thức (KPIs) thì mới đạt hiệu suất cao nhất (66.7%).
##### - Luật "Chốt sổ" (Recency Bias): Các ràng buộc nghiệp vụ (Business Rules) bắt buộc phải đặt ở vị trí cuối cùng của Prompt để ép LLM tuân thủ nghiêm ngặt luật lệ ngay trước khi sinh code.

### Future works
##### - Context Selection: Dựa trên câu hỏi, chỉ lấy KPIs, Rules, Schema liên quan để vào prompt | tối ưu độ dài, chi phí, độ chính xác
##### - Retrieval-based Few-shot: Chọn examples gần nhất với câu hỏi thay vì dùng cố định